In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)


import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import GCN2Conv


d:\elliptic\venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

DATA_PATH = r"D:\elliptic\blte\Labeled-Transactions-based-Dataset-of-Ethereum-Network-master\FinalDataset.xlsx"
df = pd.read_excel(DATA_PATH)

print("Raw shape:", df.shape)

df["from_scam"] = df["from_scam"].fillna(0).astype(int)
df["to_scam"]   = df["to_scam"].fillna(0).astype(int)

df["label"] = ((df["from_scam"] == 1) | (df["to_scam"] == 1)).astype(int)
print("Label distribution:\n", df["label"].value_counts())

df = df.reset_index(drop=True)
df["txId"] = np.arange(len(df), dtype=int)



print("Columns after dropping time cols:", df.columns.tolist())


Raw shape: (71250, 18)
Label distribution:
 label
0    57000
1    14250
Name: count, dtype: int64
Columns after dropping time cols: ['hash', 'nonce', 'transaction_index', 'from_address', 'to_address', 'value', 'gas', 'gas_price', 'input', 'receipt_cumulative_gas_used', 'receipt_gas_used', 'block_timestamp', 'block_number', 'block_hash', 'from_scam', 'to_scam', 'from_category', 'to_category', 'label', 'txId']


In [3]:

ts_col = None
for c in ["block_timestamp", "block_number"]:
    if c in df.columns:
        ts_col = c
        break


ts_raw = df[ts_col].copy()

if ts_col == "block_timestamp":
    ts_dt = pd.to_datetime(ts_raw, errors="coerce")

    ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
    ts_num = (ts_dt.view("int64") // 10**9).astype("int64")  
else:
    ts_num = pd.to_numeric(ts_raw, errors="coerce")
    ts_num = ts_num.fillna(method="ffill").fillna(method="bfill").astype("int64")

id_and_addr_cols = [
    "hash",
    "from_address",
    "to_address",
    "block_hash",
    "input",
    "txId",
    "block_timestamp",
    "block_number",
]

label_related_cols = [
    "from_scam",
    "to_scam",
    "from_category",
    "to_category",
    "label",
]

cols_to_exclude = set(id_and_addr_cols + label_related_cols)

feature_cols = [c for c in df.columns if c not in cols_to_exclude]


X_df = df[feature_cols].copy()
non_numeric = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("Bỏ cột không phải số:", non_numeric)
    X_df = X_df.drop(columns=non_numeric)

feature_cols = X_df.columns.tolist()
print("Num feature cols:", len(feature_cols))
print("Feature columns:", feature_cols)

X_all = X_df.astype(float).values
y_all = df["label"].values
txid_all = df["txId"].values

print("X_all:", X_all.shape, "y_all:", y_all.shape, "txid_all:", txid_all.shape)
print("ts_col:", ts_col, "| ts_num shape:", ts_num.shape)


Num feature cols: 7
Feature columns: ['nonce', 'transaction_index', 'value', 'gas', 'gas_price', 'receipt_cumulative_gas_used', 'receipt_gas_used']
X_all: (71250, 7) y_all: (71250,) txid_all: (71250,)
ts_col: block_timestamp | ts_num shape: (71250,)


C:\Users\Admin\AppData\Local\Temp\ipykernel_16024\1604688446.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
C:\Users\Admin\AppData\Local\Temp\ipykernel_16024\1604688446.py:14: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ts_num = (ts_dt.view("int64") // 10**9).astype("int64")


In [4]:


def build_tx_edges(df, addr_col, txid_col="txId"):

    edges = []
    for addr, group in df.groupby(addr_col):
        if len(group) <= 1:
            continue
        ids = group[txid_col].values
        src = ids[:-1]
        dst = ids[1:]
        edges.extend(zip(src, dst))
    return edges

edges_from = build_tx_edges(df, "from_address")
edges_to   = build_tx_edges(df, "to_address")

all_edges = edges_from + edges_to
print("Num raw edges:", len(all_edges))

if len(all_edges) > 0:
    edges_array = np.array(all_edges, dtype=np.int64)

    edges_array = np.sort(edges_array, axis=1)
    edges_array = np.unique(edges_array, axis=0)
else:
    edges_array = np.empty((0, 2), dtype=np.int64)

print("Num unique edges:", len(edges_array))

df_edges = pd.DataFrame(edges_array, columns=["txId1", "txId2"])

Num raw edges: 67812
Num unique edges: 65764


In [5]:


TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9

unique_ts = np.sort(np.unique(ts_num))
n_ts = len(unique_ts)

print(f"ts_col = {ts_col} | số time buckets = {n_ts}")


train_cut = int(np.floor(TRAIN_RATIO * n_ts))
val_cut   = int(np.floor((TRAIN_RATIO + VAL_RATIO) * n_ts))

train_cut = max(train_cut, 1)
val_cut   = max(val_cut, train_cut + 1)
val_cut   = min(val_cut, n_ts - 1)

train_ts = unique_ts[:train_cut]
val_ts   = unique_ts[train_cut:val_cut]
test_ts  = unique_ts[val_cut:]

train_mask = np.isin(ts_num, train_ts)
val_mask   = np.isin(ts_num, val_ts)
test_mask  = np.isin(ts_num, test_ts)

assert not np.any(train_mask & val_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(val_mask & test_mask)
assert np.all(train_mask | val_mask | test_mask)

X_train_raw, y_train, txid_train = X_all[train_mask], y_all[train_mask], txid_all[train_mask]
X_val_raw,   y_val,   txid_val   = X_all[val_mask],   y_all[val_mask],   txid_all[val_mask]
X_test_raw,  y_test,  txid_test  = X_all[test_mask],  y_all[test_mask],  txid_all[test_mask]

def show_stats(name, yy):
    counts = np.bincount(yy)
    n0 = counts[0] if len(counts) > 0 else 0
    n1 = counts[1] if len(counts) > 1 else 0
    ratio = n1 / (n0 + n1) if (n0 + n1) > 0 else 0
    print(f"{name:5s}: 0 = {n0:6d}, 1 = {n1:6d}, scam_ratio = {ratio:.6f}")


show_stats("ALL",   y_all)
show_stats("Train", y_train)
show_stats("Val",   y_val)
show_stats("Test",  y_test)

print("\nKích thước:")
print("X_train:", X_train_raw.shape, "y_train:", y_train.shape)
print("X_val  :", X_val_raw.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test_raw.shape,  "y_test :", y_test.shape)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

txid_trainval = np.concatenate([txid_train, txid_val])
X_trainval    = np.concatenate([X_train, X_val], axis=0)
y_trainval    = np.concatenate([y_train, y_val], axis=0)

print("\nAfter scaling:")
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Train+Val size:", len(X_trainval), "Train+Val pos:", int(y_trainval.sum()))


ts_col = block_timestamp | số time buckets = 28683
ALL  : 0 =  57000, 1 =  14250, scam_ratio = 0.200000
Train: 0 =  42538, 1 =   3021, scam_ratio = 0.066310
Val  : 0 =   7200, 1 =    699, scam_ratio = 0.088492
Test : 0 =   7262, 1 =  10530, scam_ratio = 0.591839

Kích thước:
X_train: (45559, 7) y_train: (45559,)
X_val  : (7899, 7) y_val  : (7899,)
X_test : (17792, 7) y_test : (17792,)

After scaling:
X_train: (45559, 7) X_val: (7899, 7) X_test: (17792, 7)
Train+Val size: 53458 Train+Val pos: 3720


In [6]:


def build_edge_index(df_edges, valid_txids):

    node_ids = np.asarray(valid_txids, dtype=np.int64)
    id2idx = {tid: i for i, tid in enumerate(node_ids)}

    mask = df_edges["txId1"].isin(node_ids) & df_edges["txId2"].isin(node_ids)
    edges_sub = df_edges.loc[mask, ["txId1", "txId2"]]

    if len(edges_sub) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    src_idx = edges_sub["txId1"].map(id2idx).values
    dst_idx = edges_sub["txId2"].map(id2idx).values
    edges_idx = np.vstack([src_idx, dst_idx])

    edge_index = torch.tensor(edges_idx, dtype=torch.long)
    edge_index = to_undirected(edge_index)
    return edge_index

edge_index_train = build_edge_index(df_edges, txid_train)
edge_index_val   = build_edge_index(df_edges, txid_val)
edge_index_test  = build_edge_index(df_edges, txid_test)

print("Train edges:", edge_index_train.size(1),
      "Val edges:", edge_index_val.size(1),
      "Test edges:", edge_index_test.size(1))


Train edges: 70760 Val edges: 11948 Test edges: 45670


In [7]:


X_train_gcn = torch.tensor(X_train, dtype=torch.float)
X_val_gcn   = torch.tensor(X_val,   dtype=torch.float)
X_test_gcn  = torch.tensor(X_test,  dtype=torch.float)

y_train_gcn = torch.tensor(y_train, dtype=torch.long)
y_val_gcn   = torch.tensor(y_val,   dtype=torch.long)
y_test_gcn  = torch.tensor(y_test,  dtype=torch.long)

train_data = Data(x=X_train_gcn, edge_index=edge_index_train, y=y_train_gcn)
val_data   = Data(x=X_val_gcn,   edge_index=edge_index_val,   y=y_val_gcn)
test_data  = Data(x=X_test_gcn,  edge_index=edge_index_test,  y=y_test_gcn)


train_data.node_ids = torch.tensor(txid_train, dtype=torch.long)
val_data.node_ids   = torch.tensor(txid_val,   dtype=torch.long)
test_data.node_ids  = torch.tensor(txid_test,  dtype=torch.long)

print(train_data)
print(val_data)
print(test_data)

print("Check NaN in train features:", torch.isnan(train_data.x).any().item())
print("Check Inf in train features:", torch.isinf(train_data.x).any().item())

Data(x=[45559, 7], edge_index=[2, 70760], y=[45559], node_ids=[45559])
Data(x=[7899, 7], edge_index=[2, 11948], y=[7899], node_ids=[7899])
Data(x=[17792, 7], edge_index=[2, 45670], y=[17792], node_ids=[17792])
Check NaN in train features: False
Check Inf in train features: False


In [8]:


if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

class_sample_count = torch.bincount(train_data.y, minlength=2).float()
eps = 1e-8
inv_freq = 1.0 / (class_sample_count + eps)
norm_inv_freq = inv_freq / inv_freq.min()

print("Class counts (train):", class_sample_count.tolist())
print("Class weights (inv_freq normalized):", norm_inv_freq.tolist())

Device: xpu
Class counts (train): [42538.0, 3021.0]
Class weights (inv_freq normalized): [1.0, 14.080768585205078]


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import scatter  


def create_ffn(in_dim: int, hidden_units: int, dropout_rate: float) -> nn.Module:

    return nn.Sequential(
        nn.Linear(in_dim, hidden_units),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate),
        nn.Linear(hidden_units, hidden_units),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate),
    )


class GraphConvLayer(nn.Module):


    def __init__(
        self,
        hidden_units: int,
        dropout_rate: float = 0.2,
        aggregation_type: str = "mean",   
        combination_type: str = "concat",   
        normalize: bool = False,
    ):
        super().__init__()
        self.hidden_units = hidden_units
        self.dropout_rate = dropout_rate
        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize


        self.ffn_prepare = create_ffn(hidden_units, hidden_units, dropout_rate)

        if self.combination_type == "gru":
            self.update_gru = nn.GRU(
                input_size=hidden_units,
                hidden_size=hidden_units,
                batch_first=True,
                dropout=dropout_rate if dropout_rate > 0 else 0.0,
            )
            self.update_mlp = None
        else:
            in_dim = 2 * hidden_units if self.combination_type == "concat" else hidden_units
            self.update_mlp = create_ffn(in_dim, hidden_units, dropout_rate)
            self.update_gru = None

    def forward(self, node_repr: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor | None):
        """
        node_repr: [N, hidden]
        edge_index: [2, E] (src -> dst)
        edge_weight: [E] or None
        """
        src, dst = edge_index[0], edge_index[1]
        neigh_repr = node_repr[src] 

        messages = self.ffn_prepare(neigh_repr)  
        if edge_weight is not None:
            messages = messages * edge_weight.view(-1, 1)


        reduce = {"sum": "sum", "mean": "mean", "max": "max"}[self.aggregation_type]
        aggregated = scatter(messages, dst, dim=0, dim_size=node_repr.size(0), reduce=reduce)  # [N, hidden]


        if self.combination_type == "gru":
            seq = torch.stack([node_repr, aggregated], dim=1)  
            out, _ = self.update_gru(seq)
            node_emb = out[:, -1, :]                        
        elif self.combination_type == "concat":
            h = torch.cat([node_repr, aggregated], dim=1)       
            node_emb = self.update_mlp(h)                   
        elif self.combination_type == "add":
            h = node_repr + aggregated                       
            node_emb = self.update_mlp(h)                        
        else:
            raise ValueError(f"Invalid combination_type: {self.combination_type}")

        if self.normalize:
            node_emb = F.normalize(node_emb, p=2, dim=-1)
        return node_emb


class GNNNodeClassifier(nn.Module):


    def __init__(
        self,
        in_features: int,
        num_classes: int,
        hidden_units: int,
        aggregation_type: str = "sum",
        combination_type: str = "concat",
        dropout_rate: float = 0.2,
        normalize: bool = True,
    ):
        super().__init__()


        self.preprocess = create_ffn(in_features, hidden_units, dropout_rate)

        self.conv1 = GraphConvLayer(
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            aggregation_type=aggregation_type,
            combination_type=combination_type,
            normalize=normalize,
        )
        self.conv2 = GraphConvLayer(
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            aggregation_type=aggregation_type,
            combination_type=combination_type,
            normalize=normalize,
        )

        self.postprocess = create_ffn(hidden_units, hidden_units, dropout_rate)
        self.compute_logits = nn.Linear(hidden_units, num_classes)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor | None = None):

        if edge_weight is None:
            edge_weight = torch.ones(edge_index.size(1), device=x.device, dtype=x.dtype)
        edge_weight = edge_weight / (edge_weight.sum() + 1e-12)

        h = self.preprocess(x)                  
        h1 = self.conv1(h, edge_index, edge_weight)  
        h = h1 + h                              
        h2 = self.conv2(h, edge_index, edge_weight) 
        h = h2 + h                               

        h = self.postprocess(h)                  
        logits = self.compute_logits(h)           
        return logits


In [10]:



class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()

        target = target.view(-1, 1)
        logp_t = logp.gather(1, target).squeeze(1)
        p_t    = p.gather(1, target).squeeze(1)

        focal_term = (1 - p_t) ** self.gamma
        loss = - focal_term * logp_t

        if self.weight is not None:
            w = self.weight[target.squeeze(1)].view(-1)
            loss = loss * w

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss


def train_one_config(config, loss_type="ce", class_weights=None,
                     max_epochs=400, patience=30, verbose=False):
    in_dim  = train_data.x.size(1)
    out_dim = 2

    model = GNNNodeClassifier(
        in_features = train_data.x.size(1),
        num_classes = 2,
        hidden_units = config["hid_dim"],    
        dropout_rate = config["dropout"],   
        aggregation_type = "mean",
        combination_type = "concat",
        normalize = True
    ).to(device)

    cw = class_weights.to(device) if class_weights is not None else None

    if loss_type == "ce":
        crit = nn.CrossEntropyLoss(weight=cw)
    elif loss_type == "focal":
        crit = FocalLoss(gamma=config.get("gamma", 2.0), weight=cw)
    else:
        raise ValueError("loss_type must be 'ce' or 'focal'")

    opt = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    def eval_for_search(data):
        model.eval()
        with torch.no_grad():
            out = model(data.x.to(device), data.edge_index.to(device))
            loss = crit(out, data.y.to(device)).item()
            preds = out.argmax(dim=1).cpu().numpy()
            y_true = data.y.cpu().numpy()
            macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        return loss, macro_f1

    best_state = None
    best_val_macro = -1.0
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(train_data.x.to(device), train_data.edge_index.to(device))
        loss_train = crit(out, train_data.y.to(device))
        loss_train.backward()
        opt.step()

        val_loss, val_macro = eval_for_search(val_data)

        if verbose and (epoch % 20 == 0 or epoch == 1):
            print(f"[{config.get('name','?')}] Epoch {epoch:03d} "
                  f"- train_loss={loss_train.item():.6f} "
                  f"- val_loss={val_loss:.6f} "
                  f"- val_macro={val_macro:.6f}")

        if val_macro > best_val_macro + 1e-4:
            best_val_macro = val_macro
            best_state = torch.save(model.state_dict(), "tmp_best_tx_model.pt")
            best_state = torch.load("tmp_best_tx_model.pt", map_location="cpu")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            if verbose:
                print(f"Early stop (no improve {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_macro

In [11]:


def _safe_roc_auc(y_true, y_score):
    """Return ROC-AUC or NaN if y_true has <2 classes."""
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return float("nan")
    return roc_auc_score(y_true, y_score)

@torch.no_grad()
def tune_threshold_on_val(model, data, thresholds=None):
    model.eval()
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 17)

    out = model(data.x.to(device), data.edge_index.to(device))
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    y_true = data.y.cpu().numpy()

    results = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1_scam = f1_score(y_true, preds, pos_label=1, zero_division=0)
        macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        results.append((t, f1_scam, macro_f1))

    best = max(results, key=lambda x: x[2])  # theo macro-F1
    best_t, best_f1_scam, best_macro = best

    print("\n=== Threshold search on VAL ===")
    for t, f1_s, f1_m in results:
        print(f"th={t:.2f}  F1_scam={f1_s:.6f}  macro_F1={f1_m:.6f}")

    print(f"\n>>> Best threshold = {best_t:.2f} "
          f"(F1_scam={best_f1_scam:.6f}, macro_F1={best_macro:.6f})")

    return best_t, results


@torch.no_grad()
def evaluate_with_threshold(model, data, threshold, name="TEST"):
    model.eval()
    out = model(data.x.to(device), data.edge_index.to(device))
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    y_true = data.y.cpu().numpy()
    auc = _safe_roc_auc(y_true, probs)
    preds = (probs >= threshold).astype(int)

    f1_scam  = f1_score(y_true, preds, pos_label=1, zero_division=0)
    micro_f1 = f1_score(y_true, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    print(f"\n{name} with threshold={threshold:.2f}")
    print(f"F1 (scam): {f1_scam:.6f}")
    print(f"Micro-F1: {micro_f1:.6f}")
    print(f"Macro-F1: {macro_f1:.6f}")
    print(f"ROC-AUC: {auc:.6f}")
    print("Confusion matrix:\n", cm)
    print("\nClassification report:")
    print(classification_report(y_true, preds, digits=6))

In [12]:


arch_space = [
    {"name": "arch1", "hid_dim": 64,  "num_layers": 8,  "dropout": 0.5,
     "lr": 1e-2, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch2", "hid_dim": 64,  "num_layers": 16, "dropout": 0.5,
     "lr": 1e-2, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch3", "hid_dim": 128, "num_layers": 16, "dropout": 0.5,
     "lr": 5e-3, "weight_decay": 5e-4, "alpha": 0.1},
    {"name": "arch4", "hid_dim": 128, "num_layers": 32, "dropout": 0.5,
     "lr": 5e-3, "weight_decay": 1e-3,"alpha": 0.1},
]

loss_configs = [
    {"name": "CE_no_weight",   "loss_type": "ce",
     "class_weights": torch.tensor([1.0, 1.0])},
    {"name": "CE_inv_freq",    "loss_type": "ce",
     "class_weights": norm_inv_freq},
    {"name": "Focal_gamma1.5", "loss_type": "focal",
     "class_weights": norm_inv_freq, "gamma": 1.5},
    {"name": "Focal_gamma2.0", "loss_type": "focal",
     "class_weights": norm_inv_freq, "gamma": 2.0},
]

best_model = None
best_arch  = None
best_loss_cfg = None
best_val_macro = -1.0

for arch in arch_space:
    for lc in loss_configs:
        cfg = arch.copy()
        cfg["name"] = arch["name"] + "_" + lc["name"]
        if lc["loss_type"] == "focal":
            cfg["gamma"] = lc["gamma"]

        print(f"\n=== Training config: {cfg['name']} (loss={lc['loss_type']}) ===")
        model_cfg, val_macro = train_one_config(
            cfg,
            loss_type=lc["loss_type"],
            class_weights=lc["class_weights"],
            max_epochs=300,
            patience=40,
            verbose=False,
        )
        print(f"Config {cfg['name']} val_macro = {val_macro:.6f}")

        if val_macro > best_val_macro:
            best_val_macro = val_macro
            best_model = model_cfg
            best_arch = arch
            best_loss_cfg = lc

print("\n>>> BEST CONFIG OVERALL")
print("Best arch:", best_arch)
print("Best loss config:", best_loss_cfg)
print("Best val macro-F1:", best_val_macro)



=== Training config: arch1_CE_no_weight (loss=ce) ===
Config arch1_CE_no_weight val_macro = 0.476853

=== Training config: arch1_CE_inv_freq (loss=ce) ===
Config arch1_CE_inv_freq val_macro = 0.679080

=== Training config: arch1_Focal_gamma1.5 (loss=focal) ===
Config arch1_Focal_gamma1.5 val_macro = 0.645641

=== Training config: arch1_Focal_gamma2.0 (loss=focal) ===
Config arch1_Focal_gamma2.0 val_macro = 0.688667

=== Training config: arch2_CE_no_weight (loss=ce) ===
Config arch2_CE_no_weight val_macro = 0.476853

=== Training config: arch2_CE_inv_freq (loss=ce) ===
Config arch2_CE_inv_freq val_macro = 0.637427

=== Training config: arch2_Focal_gamma1.5 (loss=focal) ===
Config arch2_Focal_gamma1.5 val_macro = 0.634307

=== Training config: arch2_Focal_gamma2.0 (loss=focal) ===
Config arch2_Focal_gamma2.0 val_macro = 0.656035

=== Training config: arch3_CE_no_weight (loss=ce) ===
Config arch3_CE_no_weight val_macro = 0.476853

=== Training config: arch3_CE_inv_freq (loss=ce) ===
Conf

In [13]:


best_threshold, _ = tune_threshold_on_val(best_model, val_data)

evaluate_with_threshold(best_model, train_data, best_threshold, name="TRAIN")
evaluate_with_threshold(best_model, val_data,   best_threshold, name="VAL")
evaluate_with_threshold(best_model, test_data,  best_threshold, name="TEST")


=== Threshold search on VAL ===
th=0.10  F1_scam=0.162634  macro_F1=0.081595
th=0.15  F1_scam=0.162672  macro_F1=0.081891
th=0.20  F1_scam=0.162672  macro_F1=0.081891
th=0.25  F1_scam=0.161873  macro_F1=0.081907
th=0.30  F1_scam=0.160504  macro_F1=0.082190
th=0.35  F1_scam=0.160178  macro_F1=0.084368
th=0.40  F1_scam=0.244438  macro_F1=0.509074
th=0.45  F1_scam=0.346521  macro_F1=0.620426
th=0.50  F1_scam=0.442840  macro_F1=0.688667
th=0.55  F1_scam=0.547009  macro_F1=0.757329
th=0.60  F1_scam=0.316168  macro_F1=0.639003
th=0.65  F1_scam=0.025424  macro_F1=0.489849
th=0.70  F1_scam=0.000000  macro_F1=0.476853
th=0.75  F1_scam=0.000000  macro_F1=0.476853
th=0.80  F1_scam=0.000000  macro_F1=0.476853
th=0.85  F1_scam=0.000000  macro_F1=0.476853
th=0.90  F1_scam=0.000000  macro_F1=0.476853

>>> Best threshold = 0.55 (F1_scam=0.547009, macro_F1=0.757329)

TRAIN with threshold=0.55
F1 (scam): 0.291649
Micro-F1: 0.925898
Macro-F1: 0.626277
ROC-AUC: 0.812337
Confusion matrix:
 [[41488  1050]


In [14]:

txid_trainval = np.concatenate([txid_train, txid_val])
X_trainval    = np.vstack([X_train, X_val])
y_trainval    = np.concatenate([y_train, y_val])

print("Train+Val shapes:", X_trainval.shape, y_trainval.shape, txid_trainval.shape)

intersect = np.intersect1d(txid_train, txid_val)
print("Overlap txId(train, val) =", intersect.size)


edge_index_trainval = build_edge_index(df_edges, txid_trainval)
print("Train+Val edges:", edge_index_trainval.size(1))

X_trainval_gcn = torch.tensor(X_trainval, dtype=torch.float)
y_trainval_gcn = torch.tensor(y_trainval, dtype=torch.long)

trainval_data = Data(x=X_trainval_gcn, edge_index=edge_index_trainval, y=y_trainval_gcn)
trainval_data.node_ids = torch.tensor(txid_trainval, dtype=torch.long)

print("trainval_data:", trainval_data)
print("Check NaN in trainval features:", torch.isnan(trainval_data.x).any().item())
print("Check Inf in trainval features:", torch.isinf(trainval_data.x).any().item())


Train+Val shapes: (53458, 7) (53458,) (53458,)
Overlap txId(train, val) = 0
Train+Val edges: 84276
trainval_data: Data(x=[53458, 7], edge_index=[2, 84276], y=[53458], node_ids=[53458])
Check NaN in trainval features: False
Check Inf in trainval features: False


In [15]:

best_threshold_pre, _ = tune_threshold_on_val(best_model, val_data)
print("Saved best_threshold (from VAL pre-final-train):", best_threshold_pre)

_train_backup, _val_backup = train_data, val_data


train_data = trainval_data
val_data   = trainval_data   

final_model, _ = train_one_config(
    best_arch,
    loss_type=best_loss_cfg["loss_type"],
    class_weights=best_loss_cfg["class_weights"],
    max_epochs=500,
    patience=40,
    verbose=True,
)


train_data, val_data = _train_backup, _val_backup

evaluate_with_threshold(final_model, test_data, best_threshold_pre, name="TEST (final train on train+val)")



=== Threshold search on VAL ===
th=0.10  F1_scam=0.162634  macro_F1=0.081595
th=0.15  F1_scam=0.162672  macro_F1=0.081891
th=0.20  F1_scam=0.162672  macro_F1=0.081891
th=0.25  F1_scam=0.161873  macro_F1=0.081907
th=0.30  F1_scam=0.160504  macro_F1=0.082190
th=0.35  F1_scam=0.160178  macro_F1=0.084368
th=0.40  F1_scam=0.244438  macro_F1=0.509074
th=0.45  F1_scam=0.346521  macro_F1=0.620426
th=0.50  F1_scam=0.442840  macro_F1=0.688667
th=0.55  F1_scam=0.547009  macro_F1=0.757329
th=0.60  F1_scam=0.316168  macro_F1=0.639003
th=0.65  F1_scam=0.025424  macro_F1=0.489849
th=0.70  F1_scam=0.000000  macro_F1=0.476853
th=0.75  F1_scam=0.000000  macro_F1=0.476853
th=0.80  F1_scam=0.000000  macro_F1=0.476853
th=0.85  F1_scam=0.000000  macro_F1=0.476853
th=0.90  F1_scam=0.000000  macro_F1=0.476853

>>> Best threshold = 0.55 (F1_scam=0.547009, macro_F1=0.757329)
Saved best_threshold (from VAL pre-final-train): 0.55
[arch1] Epoch 001 - train_loss=0.336555 - val_loss=0.333559 - val_macro=0.065060
[a

In [16]:

import os, json
import joblib
import numpy as np
import torch
import torch.nn.functional as F

GNN_SAVE_DIR = "gnn_saved"
os.makedirs(GNN_SAVE_DIR, exist_ok=True)


ckpt = {
    "model_class": final_model.__class__.__name__,
    "state_dict": final_model.state_dict(),
    "best_arch": best_arch,
    "best_loss_cfg": best_loss_cfg,
    "feature_cols": feature_cols,
    "threshold_from_val": float(best_threshold_pre),
}
ckpt_path = os.path.join(GNN_SAVE_DIR, "blte_gnn_checkpoint.pt")
torch.save(ckpt, ckpt_path)
print("Saved checkpoint:", ckpt_path)

print("Saved scalers (*.pkl) into", GNN_SAVE_DIR)


final_model.eval()
with torch.no_grad():
    out = final_model(test_data.x.to(device), test_data.edge_index.to(device))
    proba_test = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

txid_test_saved = test_data.node_ids.detach().cpu().numpy().astype(int)

pred_path = os.path.join(GNN_SAVE_DIR, "gnn_test_preds.npz")
np.savez_compressed(
    pred_path,
    txid=txid_test_saved,
    proba=proba_test,
    threshold=float(best_threshold_pre),
)
print("Saved TEST preds:", pred_path, "shape:", proba_test.shape)


Saved checkpoint: gnn_saved\blte_gnn_checkpoint.pt
Saved scalers (*.pkl) into gnn_saved
Saved TEST preds: gnn_saved\gnn_test_preds.npz shape: (17792,)
